In [2]:
import sys, time, glob, os
W = os.path.dirname(sorted(glob.glob("/kaggle/input/**/*.whl", recursive=True))[0])
t0 = time.time()
!{sys.executable} -m pip install --no-index --find-links={W} vllm torchvision
!{sys.executable} -m pip uninstall -y torchaudio
print(f"설치 {time.time()-t0:.0f}초")

Looking in links: /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels
Processing /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels/vllm-0.27.1-cp38-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels/transformers-5.15.1-py3-none-any.whl (from vllm)
Processing /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels/protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (from vllm)
Processing /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels/starlette-1.6.0-py3-none-any.whl (from vllm)
Processing /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels/prometheus_fastapi_instrumentator-8.1.0-py3-none-any.whl (from vllm)
Processing /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels/lm_format_enforcer-0.11.3-py3-none-any.whl (from vllm)
Processing /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels/llguidance-1.7.6-cp39-abi3-manylinux_2_31_x86_64.whl (from vllm)
Processing /kaggle/input/datasets/gamaius/qwen-probe/vllm_wheels/outlines_core-

In [6]:
# ====================================================================
#  셀 2 — 경로 확정 · 사전 점검 · 작업폴더 청소
# ====================================================================
import glob, os, shutil, subprocess, sys
import pandas as pd

# ── 작업폴더 청소 ───────────────────────────────────────────────────
for p in (glob.glob("/kaggle/working/sub_*.csv") + glob.glob("/kaggle/working/bk*")
          + glob.glob("/kaggle/working/submission*.csv")):
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)

# ── 입력 CSV ───────────────────────────────────────────────────────
TEST = glob.glob("/kaggle/input/**/test_submission.csv", recursive=True)
assert len(TEST) == 1, f"test_submission.csv 가 {len(TEST)}개: {TEST}"
TEST, = TEST
ref = pd.read_csv(TEST)
assert {"id", "question"} <= set(ref.columns), ref.columns.tolist()

# ── 추론 스크립트 ──────────────────────────────────────────────────
SC = glob.glob("/kaggle/input/**/02_infer_vllm.py", recursive=True)
assert len(SC) == 1, f"02_infer_vllm.py 가 {len(SC)}개: {SC}"
SC, = SC

# ── 모델 ─────────────────────────────────────────────
CANDS = [os.path.dirname(c)
         for c in glob.glob("/kaggle/input/**/config.json", recursive=True)
         if glob.glob(os.path.dirname(c) + "/*.safetensors")]
print("모델 후보:")
for c in CANDS:
    print("   ", c)

M = [c for c in CANDS if "qwen_v4_merged" in c]
assert len(M) == 1, f"\n★ qwen_v4_merged 를 특정할 수 없습니다. 후보: {CANDS}"
M, = M

GB = sum(os.path.getsize(f) for f in glob.glob(M + "/*.safetensors")) / 2**30
assert GB > 4, f"가중치가 {GB:.2f} GiB — 파일이 불완전합니다"

# ── vLLM 사전 점검 ─────────────────────────────────────────────────
r = subprocess.run([sys.executable, "-c", "import vllm; print(vllm.__version__)"],
                   capture_output=True, text=True)
assert r.returncode == 0, f"vllm 없음 — 셀 1부터 다시:\n{r.stderr[-400:]}"

# ── 확인 ───────────────────────────────────────────────────────────
print(f"""
{'='*60}
  입력   {TEST}
         {len(ref)}행 | 열 {list(ref.columns)}
  스크립트 {SC}
  모델   {M}
         {GB:.2f} GiB
  vllm   {r.stdout.strip()}
  GPU    {os.popen('nvidia-smi -L').read().strip()}
{'='*60}""")

문제 본문 겹침: 2 / 831

2000쪽 id 예시: ['test-0000', 'test-0001', 'test-0002']
831쪽  id 예시: ['val-000000', 'val-000001', 'val-000002']

2000쪽 첫 문제: The average mark of the students of a class in a particular exam is some value. If 5 students whose average mark in that


In [ ]:
N, TEMP, MT, MML = "32", "0.5", "2048", "3072"

ps = [subprocess.Popen(
        [sys.executable, "-u", SC, "--model", M, "--input", TEST,
         "--output", f"/kaggle/working/sub_{i}.csv",
         "--shard", str(i), "--num-shards", "2",
         "--n", N, "--temperature", TEMP,
         "--max-tokens", MT, "--max-model-len", MML,
         "--chunk", "50", "--backup-dir", f"/kaggle/working/bk{i}"],
        env={**os.environ, "CUDA_VISIBLE_DEVICES": str(i)},
        stdout=open(f"/kaggle/working/lb_{i}.txt", "w"),
        stderr=subprocess.STDOUT)
      for i in range(2)]
assert [p.wait() for p in ps] == [0, 0], "샤드 실패 — lb_0.txt / lb_1.txt 확인"

sub = (pd.concat([pd.read_csv(f"/kaggle/working/sub_{i}.csv") for i in range(2)])
         .drop_duplicates("id", keep="last"))
sub = ref[["id"]].merge(sub[["id", "answer"]], on="id")   # 원본 순서 유지

assert len(sub) == len(ref), f"{len(sub)}행 — {len(ref)}이어야 함"
assert set(sub["id"]) == set(ref["id"]), "id 집합 불일치"
assert sub["answer"].notna().all(), "빈 답 존재"
sub["answer"] = sub["answer"].astype("int64")
sub.to_csv("/kaggle/working/submission.csv", index=False)

for i in range(2):
    shutil.rmtree(f"/kaggle/working/bk{i}", ignore_errors=True)
    os.remove(f"/kaggle/working/sub_{i}.csv")
print(f"완료 {len(sub)}행 | 범위 {sub['answer'].min()} ~ {sub['answer'].max()}")
for i in range(2):
    print(f"\n--- shard {i} ---")
    print("".join(open(f"/kaggle/working/lb_{i}.txt").readlines()[-12:]))